# Connect Your Development Environment to MLflow

This guide shows you how to connect your development environment to an MLflow experiment. You can run MLflow:

- **Locally**, with the client and server on the same machine (this notebook).
- **Self-hosted**, on a shared server somewhere on your network or in the cloud.
- **Managed**, e.g. [Databricks Managed MLflow](https://mlflow.org/docs/latest/getting-started/databricks-trial.html).

This notebook focuses on the **local OSS MLflow** path, which is what the rest of the tutorials in this repo assume. It's a slight modification of the [official setup guide](https://mlflow.org/docs/latest/ml/getting-started/running-notebooks/). Anywhere this notebook corrects, supplements, or diverges from upstream, the relationship is named with an inline bold callout — one of **Bug**, **Stale**, **Missing from**, or **Diverges from upstream tutorial:** — or, in code cells, with a `NOTE (differs from upstream)` comment. Sections with no upstream counterpart use a plain heading and no callout.

## Prerequisites

Two requirements before any cell in this notebook will work:

- **Python 3.14+** — pinned by `requires-python = ">=3.14"` in `pyproject.toml`. If you're on an older Python, `uv sync` will refuse to resolve and the notebook kernel won't start.
- **The repo's virtual environment activated.** Without it, `import mlflow` either fails (no system-wide install) or imports an unrelated MLflow version from another project — leaving you to debug mysterious version mismatches. With `direnv` installed, `.envrc` activates the venv automatically when you `cd` into the repo; otherwise run `source .venv/bin/activate` from the repo root.

## Step 1: Install MLflow

**Diverges from upstream tutorial:** Upstream says `pip install --upgrade "mlflow>=3.1"`. This repo uses **`uv` exclusively** for dependency management — `pip` is forbidden, including `uv pip install` (see `CLAUDE.md`). MLflow is already in `pyproject.toml`, so a fresh checkout only needs:

```bash
uv sync
```

Adding or upgrading a dependency goes through the project workflow so `pyproject.toml` and `uv.lock` stay the single source of truth:

```bash
uv add 'mlflow>=3.1'             # add or pin
uv add --upgrade-package mlflow  # upgrade an already-pinned package
```

**Databricks variant:** if you ever connect this notebook to Databricks Managed MLflow, install the `[databricks]` extra: `uv add 'mlflow[databricks]>=3.1'`. For the local-only path covered here it isn't needed.

## Step 2: Configure tracking

Every MLflow client call needs to know *where* to log to. That "where" is the **tracking URI**. There are three common shapes:

| Option | Tracking URI on the client | What runs locally |
|---|---|---|
| **A — Database (recommended for local dev)** | `sqlite:///mlflow.db` | Nothing extra; the client writes directly to the SQLite file. |
| **B — File system** *(legacy, KTLO mode)* | `file:./mlruns` or unset | Nothing extra; the client writes flat files into `./mlruns/`. |
| **C — Remote tracking server** | `http://<host>:<port>` | A running `mlflow server` process the client talks to over HTTP. |

The rest of the tutorials in this repo use **Option C** — they expect a tracking server running on `http://127.0.0.1:5000`. We'll set that up after the next two sections.

> **A note on Option B (file-system backend):** the upstream docs label this **"TO BE DEPRECATED SOON"** — it doesn't support concurrent writes and, critically, **does not support the Model Registry**. Calls like `mlflow.register_model(...)` or `registered_model_name=...` simply fail against a file backend. Treat Option B as a curio; use Option A or C for anything you care about.

## `mlflow ui` vs `mlflow server`

Reading MLflow docs, Stack Overflow answers, and AI-generated summaries, you'll see both `mlflow ui` and `mlflow server` used to start the local tracking server, and many sources describe them as different commands. A typical (and **incorrect** for MLflow 3.x) framing looks like:

> *"`mlflow ui` is for local, quick development on a single machine, while `mlflow server` is geared toward production, allowing remote access, artifact storage, and handling multiple users."*

**In MLflow 3.x, that distinction no longer exists. `mlflow ui` is a string-level alias for `mlflow server`** — same defaults (`--host 127.0.0.1`, `--port 5000`, `--workers 4`), same flags, same Gunicorn-backed server. Anything you can do with one you can do with the other by changing flags.

### Evidence 1: the CLI helps are byte-identical

```bash
$ diff <(mlflow ui --help) <(mlflow server --help)
1c1
< Usage: mlflow ui [OPTIONS]
---
> Usage: mlflow server [OPTIONS]
```

Only the program name on line 1 differs. Every flag, every default, every help paragraph is the same.

### Evidence 2: the source code makes the equivalence explicit

From `mlflow/cli/__init__.py` in mlflow `3.12.0` (lines 63–67):

```python
class AliasedGroup(click.Group):
    def get_command(self, ctx, cmd_name):
        # `mlflow ui` is an alias for `mlflow server`
        cmd_name = "server" if cmd_name == "ui" else cmd_name
        return super().get_command(ctx, cmd_name)
```

Click rewrites `ui` → `server` *before* dispatch. There is one implementation; the `ui` name never reaches a different code path. So claims like "`mlflow ui` doesn't support Postgres", "`mlflow ui` only runs one worker", or "`mlflow ui` binds to 127.0.0.1 *unlike* `mlflow server`" are all false on MLflow 3.x — both commands accept the same `--backend-store-uri`, the same `--workers`, and both default to `--host 127.0.0.1`.

### So when do people use which name?

Purely convention now:

- `mlflow ui` reads as "give me the local UI for my experiments" — common in quickstart docs and tutorials.
- `mlflow server` reads as "start the tracking service" — common in deployment scripts, systemd units, Docker entrypoints, and team-shared setups.

Pick whichever name fits the *intent* the reader will infer; under the hood you get the same process either way. This repo's notebooks use `mlflow server` in setup instructions because we're treating the local tracking server as a "service" the notebooks talk to over HTTP.

### Historical context (pre-MLflow 3)

The "ui = quick local, server = production" framing was **accurate at one point**, which is why it's still circulating. In older MLflow versions:

- `mlflow ui` ran a single-process Flask dev server, defaulted to the file backend (`./mlruns`), and didn't expose `--workers` or many of the production flags.
- `mlflow server` ran under Gunicorn with multiple workers, accepted `--workers` / `--gunicorn-opts`, and was the documented entry point for multi-user deployments.

If you find a blog post or AI answer that describes the two as different, check its date (or its training cutoff). The unification happened in MLflow 3.

## Client tracking URI vs server backend store URI

The single most common source of "my run vanished!" confusion in MLflow is mixing up two different URIs that both sound like "where MLflow stores things":

| URI | Lives on | Set with | What it answers |
|---|---|---|---|
| **Tracking URI** | The **client** (your notebook / training script) | `mlflow.set_tracking_uri(...)` or `MLFLOW_TRACKING_URI` env var | *Where do I send my log_param / log_metric / log_model calls?* |
| **Backend store URI** | The **server** (`mlflow server` process) | `--backend-store-uri` flag | *Where does the server persist what it receives?* |

These are independent. A few illustrative combinations:

**Local-only, Option A (no server, direct SQLite):**
```python
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # client writes directly to sqlite
```
There is no server. The client *is* the backend.

**Local-only, Option C (server in front of SQLite):**
```bash
mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000
```
```python
mlflow.set_tracking_uri("http://127.0.0.1:5000")  # client talks to server over HTTP
```
The server persists to SQLite; the client doesn't know or care what backend the server chose.

**Mismatched defaults — the classic foot-gun:**
```bash
# Terminal 1
mlflow server --port 5000
```
```python
# Notebook
mlflow.set_tracking_uri("sqlite:///mlflow.db")  # NOT the server!
```
The notebook writes runs directly into `mlflow.db`. The server is also using `mlflow.db` by default (in MLflow 3), so if both processes run from the **same** working directory you'll *see* the runs in the UI — but they got there bypassing the HTTP layer, which means any server-side feature (auth, artifact proxy, audit logging) is silently skipped.

**The rule:** when running a server, point the client at `http://...`, not at the same database. Anything else is a coincidence-driven setup that breaks the moment you change the cwd, add auth, or move to a remote backend.

## What the server's default backend really is

**Stale in upstream tutorial:** The `mlflow server --backend-store-uri` help text says:

> By default, data will be logged to the `./mlruns` directory.

Since [PR #18497](https://github.com/mlflow/mlflow/pull/18497) (MLflow 3), the actual default is:

- If `./mlruns/` already exists in the cwd → keep using the file backend (`file:./mlruns`) for backward compatibility.
- Otherwise → create and use **`sqlite:///mlflow.db`** in the cwd.

Why this matters here: when you start `mlflow server --port 5000` from this repo's root (no existing `mlruns/`), you get a SQLite-backed server that supports the Model Registry, transactions, and everything else the file backend can't do — **with no flags required**. The whole point of Option A's `--backend-store-uri sqlite:///mlflow.db` arg in the upstream tutorial is now the default.

## Step 3: Start the tracking server (Option C)

Open a separate terminal at the **repo root** (so `mlflow.db` and `mlartifacts/` land predictably) and run:

```bash
mlflow server --host 127.0.0.1 --port 5000
```

You should see startup logs like:

```text
Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only).
```

Now point this notebook at it and create (or re-attach to) an experiment.

In [ ]:
import mlflow

HOST = "127.0.0.1"
PORT = 5000
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

mlflow.set_experiment("my-first-experiment")

## Step 4: Verify the connection

Run a one-shot logging call. If the server is up, the run shows up in the UI and the cell prints a success line. If the server is *not* up, the cell raises a connection error pointing at port 5000 — that's the obvious failure mode to recognize.

In [ ]:
print(f"MLflow Tracking URI: {mlflow.get_tracking_uri()}")
print(f"Active Experiment:   {mlflow.get_experiment_by_name('my-first-experiment')}")

with mlflow.start_run(run_name="connection_check") as run:
    mlflow.log_param("test_param", "test_value")
    print(f"\nLogged run_id: {run.info.run_id}")
    print("Successfully connected to MLflow!")

## Step 5: Access the MLflow UI

Browse to <http://127.0.0.1:5000/> in your browser (same host and port the `mlflow server` process is listening on). You should see:

- An **Experiments** sidebar with `Default` and `my-first-experiment`.
- One run named `connection_check` under `my-first-experiment`, with the `test_param = test_value` parameter logged.

### "Invalid Host header" when accessing from another machine

If you started the server with `--host 0.0.0.0` to expose it on your LAN and the UI returns:

> Invalid Host header — possible DNS rebinding attack detected

that's MLflow 3's built-in security middleware refusing requests with a non-localhost `Host:` header. The fix is to allowlist the hostnames you'll connect from when you start the server:

```bash
mlflow server \
    --host 0.0.0.0 \
    --port 5000 \
    --allowed-hosts 127.0.0.1,my-laptop.local \
    --cors-allowed-origins http://my-laptop.local:5000
```

For purely local development, `--host 127.0.0.1` (the default) avoids the issue entirely.

## What's next

Now that the server is up and the client can reach it:

- **`b_tracking_quickstart.ipynb`** — the 5-minute tour: train a model, log it, register it, load it back as a pyfunc.
- **`c_hyperparameter_tuning.ipynb`** — the same setup, plus parent/child runs and Optuna for tuning.

### Other connection paths (out of scope for this repo)

- [Databricks Managed MLflow](https://mlflow.org/docs/latest/getting-started/databricks-trial.html) — set `MLFLOW_TRACKING_URI=databricks` plus `DATABRICKS_HOST` / `DATABRICKS_TOKEN`. The upstream tutorial covers this in its "Databricks - Local IDE" tab.
- [Self-hosting a shared tracking server](https://mlflow.org/docs/latest/self-hosting.md) — put `mlflow server` behind a reverse proxy with auth, point a Postgres backend at it, and share with your team.